# Preprocessing Text for LLMs

Based on Chapter 2 of *<a href="https://github.com/rasbt/LLMs-from-scratch">Build a Large Language Model from Scratch</a>* by Sebastian Raschka

### Reading a text file from the web

We will use the short story *The Verdict* by Edith Wharton

In [ ]:
import urllib.request

url = ("https://raw.githubusercontent.com/rasbt/"
       "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
       "the-verdict.txt")

urllib.request.urlretrieve(url, "the-verdict.txt")

# load the file contents as a single string
with open("the-verdict.txt", "r", encoding="utf-8") as fp:
    raw_text = fp.read()

In [ ]:
len(raw_text)

In [ ]:
raw_text[:99]

### A very simple tokenizer

In [ ]:
import re
tokens = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
# remove whitespace from tokens
tokens = [item.strip() for item in tokens if item.strip()]

In [ ]:
len(tokens)

In [ ]:
print(tokens[:50])

In [ ]:
# list of all unique tokens plus special marker symbols
all_tokens = sorted(set(tokens))
all_tokens.append("<|endoftext|>")
all_tokens.append("<|unknown|>")

In [ ]:
len(all_tokens)

In [ ]:
print(all_tokens[:50])

In [ ]:
# vocab is a dictionary that maps tokens to ID numbers
vocab = {t:i for i,t in enumerate(all_tokens)}

In [ ]:
vocab['!']

In [ ]:
vocab['the']

In [ ]:
vocab['<|unknown|>']

In [ ]:
class SimpleTokenizer:
    
    def __init__(self, vocab):
        self.token_to_id = vocab
        self.id_to_token = {i:t for t,i in vocab.items()}

    def encode(self, text):
        tokens = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        tokens = [item.strip() for item in tokens if item.strip()] # remove whitespace
        tokens = [t if t in self.token_to_id else "<|unknown|>" for t in tokens] # replace unknowns
        ids = [self.token_to_id[token] for token in tokens]
        return ids

    def decode(self, ids):
        text = " ".join([self.id_to_token[i] for i in ids])
        # remove spaces before the specified punctuation
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [ ]:
simple_tokenizer = SimpleTokenizer(vocab)

In [ ]:
ids = simple_tokenizer.encode(raw_text)

In [ ]:
print(ids[:50])

In [ ]:
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
combined_texts = " <|endoftext|> ".join((text1, text2))
print(combined_texts)

In [ ]:
ids = simple_tokenizer.encode(combined_texts)
print(ids)

In [ ]:
print(simple_tokenizer.decode(ids))

### Byte-pair encoding with tiktoken

In [ ]:
import tiktoken

In [ ]:
BPE_tokenizer = tiktoken.get_encoding("gpt2")

In [ ]:
# total vocabulary size
BPE_tokenizer.n_vocab

In [ ]:
BPE_tokenizer.encode("the")

In [ ]:
BPE_tokenizer.encode("snorkel")

In [ ]:
def show_ids(text):
    for i in BPE_tokenizer.encode(text):
        token = BPE_tokenizer.decode([i])
        print(f"{i} -> '{token}'")

In [ ]:
show_ids("snorkel")

In [ ]:
BPE_tokenizer.decode([82, 13099, 7750])

Here is the full short story encoded as a sequence of token IDs:

In [ ]:
ids = BPE_tokenizer.encode(raw_text)

In [ ]:
len(ids)

In [ ]:
print(ids[:50])

In [ ]:
BPE_tokenizer.decode(ids[:50])

### Data sampling with a sliding window

In [ ]:
encoded_sample = ids[50:]

In [ ]:
context_size = 5
x = encoded_sample[:context_size]
y = encoded_sample[1:context_size+1]
print(f"x: {x}")
print(f"y:      {y}")

Create the input-target pairs for training:

In [ ]:
for i in range(1, context_size+1):
    context = encoded_sample[:i]
    desired = encoded_sample[i]
    print(context, "---->", desired)

Here are the corresponding decoded tokens:

In [ ]:
for i in range(1, context_size+1):
    context = encoded_sample[:i]
    desired = encoded_sample[i]
    print(BPE_tokenizer.decode(context), "---->", BPE_tokenizer.decode([desired]))

A GPTDataset object keeps track of the input-target pairs:

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDataset(Dataset):
    
    def __init__(self, text, tokenizer, context_size, stride):
        self.token_ids = tokenizer.encode(text)
        self.input_tensors = []
        self.target_tensors = []

        for i in range(0, len(self.token_ids) - context_size, stride):
            input_chunk = self.token_ids[i:i + context_size]
            target_chunk = self.token_ids[i + 1: i + context_size + 1]
            self.input_tensors.append(torch.tensor(input_chunk))
            self.target_tensors.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_tensors)

    def __getitem__(self, idx):
        return self.input_tensors[idx], self.target_tensors[idx]


In [ ]:
dataset = GPTDataset(raw_text, BPE_tokenizer, context_size=5, stride=2)

In [ ]:
print(dataset.token_ids[:15])

In [ ]:
len(dataset)

In [ ]:
dataset[0]

In [ ]:
dataset[1]

In [ ]:
dataset[2]

Now we create a data loader to automatically break the dataset into batches:

In [ ]:
# drop_last=True drops the last batch if it is shorter than the
# specified batch_size to prevent loss spikes during training.

def create_dataloader(text, batch_size=4, context_size=256, stride=128,
                      shuffle=True, drop_last=True, num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDataset(text, tokenizer, context_size, stride)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                            drop_last=drop_last, num_workers=num_workers)
    return dataloader

In [ ]:
dataloader = create_dataloader(raw_text, batch_size=8, context_size=5, stride=2, shuffle=False)

We an iterator object that returns the next batch when Python's `next` function is called:

In [ ]:
dataloader_iter = iter(dataloader)

In [ ]:
input_batch, target_batch = next(dataloader_iter)

In [ ]:
print("Input batch:")
print(input_batch)
print("\nTarget batch:")
print(target_batch)

In [ ]:
next(dataloader_iter)

In [ ]:
input_batch.shape

In [ ]:
dataloader = create_dataloader(raw_text)

In [ ]:
dataloader_iter = iter(dataloader)

In [ ]:
next(dataloader_iter)

In [ ]:
input_batch, target_batch = next(dataloader_iter)

In [ ]:
input_batch.shape

### Creating token embeddings

In this example, we will start by representing each token in the vocabulary as a floating-point vector of length 3 (called an *embedding*). We first create an Embedding layer and retrieve its (randomly initialized) weight matrix:

In [ ]:
vocab_size = 6
output_dim = 3

torch.manual_seed(123) # for reproducibility

embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
print(embedding_layer.weight)

This tensor contains the embeddings we will use for all 6 vocabulary tokens. We can retrieve a specific embedding by applying the Embedding object to the index number of the token we want (represented as a tensor). For example, here is the embedding vector for token #3 in the vocabulary, which is simply the 3rd row of the weight matrix:

In [ ]:
embedding_layer(torch.tensor(3))

We can easily retrieve several embeddings at once:

In [ ]:
embedding_layer(torch.tensor([2, 3, 5, 1]))

Using our original vocabulary for *The Verdict*, let's create an embedding vector of size 8 for the token "the":

In [ ]:
BPE_tokenizer.n_vocab

In [ ]:
embedding_layer = torch.nn.Embedding(50257, 8)

In [ ]:
BPE_tokenizer.encode('the')

In [ ]:
embedding_layer(torch.tensor(1169))

Here are the embeddings for the first five tokens in the story:

In [ ]:
ids[:5]

In [ ]:
embedding_layer(torch.tensor(ids[:5]))

In [ ]:
# this setting makes the output more readable
torch.set_printoptions(sci_mode=False)

In [ ]:
embedding_layer(torch.tensor(ids[:5]))

And here is the full short story represented as a sequence of token embeddings:

In [ ]:
embedding_layer(torch.tensor(ids))

In [ ]:
embedding_layer(torch.tensor(ids)).shape

### Encoding token positions

In the same way we encoded each token ID in the sequence above as a floating-point vector, we will also encode information about where each token occurs in the sequence as vectors. Conceptually, we represent the position number of a token in the sequence as a one-hot vector, and then convert this one-hot vector to an embedding. From the text of *The Verdict*, we created a vocabulary of 50257 unique tokens. This time we will make our embedding vectors for both the token IDs and the token positions be of size 256.

In [ ]:
vocab_size = 50257
output_dim = 256

#### Token embeddings

First we create the embeddings for the token IDs:

In [ ]:
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
print(token_embedding_layer.weight)

In [ ]:
token_embedding_layer.weight.shape

#### Position embeddings

Next we create the positional embedding vectors. The position numbers run from 0 to `context_size-1`, and correspond to the positions of each token in the current context window. Conceptually, we represent each position number as a one-hot vector, and then convert this vector to an embedding.

In [ ]:
context_size

In [ ]:
position_embedding_layer = torch.nn.Embedding(context_size, output_dim)

In [ ]:
position_embeddings = position_embedding_layer(torch.arange(context_size))

In [ ]:
torch.arange(context_size)

In [ ]:
position_embeddings

In [ ]:
# this vector of size 256 represents position 0 of the context window
position0_embedding = position_embedding_layer(torch.tensor(0))
position0_embedding

In [ ]:
# this vector of size 256 represents token ID number 40
token40_embedding = token_embedding_layer(torch.tensor(40))
token40_embedding

In [ ]:
# the sum of these vectors represents token ID 40 at position 0 in the context window
combined_embedding = token40_embedding + position0_embedding
combined_embedding

### Putting it all together

In [ ]:
dataloader = create_dataloader(raw_text, batch_size=8, context_size=5, stride=5, shuffle=False)
dataloader_iter = iter(dataloader)

In [ ]:
input_batch, target_batch = next(dataloader_iter)
print(input_batch)
print(input_batch.shape)

In [ ]:
token_embeddings = token_embedding_layer(input_batch)

In [ ]:
print(token_embeddings.shape)

Here is the first input example in the batch, with its five token embeddings:

In [ ]:
token_embeddings[0]

In [ ]:
token_embeddings[0].shape

We now combine this with `position_embeddings` to encode the positional information for each token:

In [ ]:
print(position_embeddings.shape)

In [ ]:
token_embeddings[0] + position_embeddings

In [ ]:
(token_embeddings[0] + position_embeddings).shape

We can do this to all 8 examples in the batch at once, using PyTorch's tensor *broadcasting* operation:

In [ ]:
final_input_batch = token_embeddings + position_embeddings
final_input_batch

In [ ]:
final_input_batch.shape

### A simple example of broadcasting in PyTorch

In [ ]:
t = torch.tensor([[1,2,3],
                  [4,5,6],
                  [7,8,9],
                  [10,11,12]])
print(t)
print(t.shape)

In [ ]:
inc = torch.tensor([10,10,10])

In [ ]:
print(inc)
print(inc.shape)

In [ ]:
s = t + inc
print(s)
print(s.shape)